***Create an adjustable mapping matrix from fine to coarse grain representations***

In [2]:
import torch
import mdtraj as md
import numpy as np
import os
import json

In [ ]:
#main path to the project folder
main_path="../.."

Create Groupings


In [4]:
#Create various center of mass groupings of Alanine Dipeptide by combining the atoms by their indices
ramachandran_COM=[[0,1,2,3,4,5],
              [6,7],
              [8,9,10,11,12,13],
              [14,15],
              [16,17,18,19,20,21]]
ramachandran_list_COM=["ramachandran_COM",ramachandran_COM]

ramachandran_beta_COM=[[0,1,2,3,4,5],
              [6,7],
              [8,9],
              [10,11,12,13],        #beta c-atom group for chirality
              [14,15],
              [16,17,18,19,20,21]]
ramachandran_beta_list_COM=["ramachandran_beta_COM",ramachandran_beta_COM]

backbone_COM=[[0,1,2,3],
          [4,5],
          [6,7],
          [8,9,10,11,12,13],
          [14,15],
          [16,17],
          [18,19,20,21]]
backbone_list_COM=["backbone_COM",backbone_COM]

backbone_beta_COM=[[0,1,2,3],
          [4,5],
          [6,7],
          [8,9],
          [10,11,12,13],
          [14,15],
          [16,17],
          [18,19,20,21]]
backbone_beta_list_COM=["backbone_beta_COM",backbone_beta_COM]

short_COM=[[0,1,2,3,4,5,6,7],
              [8,9,10,11,12,13],
              [14,15,16,17,18,19,20,21]]
short_list_COM=["short_COM",short_COM]

short_beta_COM=[[0,1,2,3,4,5,6,7],
              [8,9],
              [10,11,12,13],
              [14,15,16,17,18,19,20,21]]
short_beta_list_COM=["short_beta_COM",short_beta_COM]


In [ ]:
#Create coarse grained mappings of sliced atoms: central atom for latent grouping
ramachandran_slice=[[4,[0,1,2,3,5]],
              [6,[7]],
              [8,[9,10,11,12,13]],
              [14,[15]],
              [16,[17,18,19,20,21]]]
ramachandran_list_slice=["ramachandran_Slice",ramachandran_slice]

ramachandran_beta_slice=[[4,[0,1,2,3,5]],
              [6,[7]],
              [8,[9]],
              [10,[11,12,13]],        #beta c-atom group for chirality
              [14,[15]],
              [16,[17,18,19,20,21]]]
ramachandran_beta_list_slice=["ramachandran_beta_Slice",ramachandran_beta_slice]

backbone_slice=[[0,[1,2,3]],
          [4,[5]],    
          [6,[7]],
          [8,[9,10,11,12,13]],        #beta c-atom group for chirality
          [14,[15]],
          [16,[17]],
          [18,[19,20,21]]]
backbone_list_slice=["backbone_Slice",backbone_slice]

backbone_beta_slice=[[0,[1,2,3]],
          [4,[5]],    
          [6,[7]],
          [8,[9]],
          [10,[11,12,13]],        #beta c-atom group for chirality
          [14,[15]],
          [16,[17]],
          [18,[19,20,21]]]
backbone_beta_list_slice=["backbone_beta_Slice",backbone_beta_slice]

short_slice=[[6,[0,1,2,3,4,5,7]],
              [8,[9,10,11,12,13]],
              [14,[15,16,17,18,19,20,21]]]
short_list_slice=["short_Slice",short_slice]

short_beta_slice=[[6,[0,1,2,3,4,5,7]],
              [8,[9]],
              [10,[11,12,13]],
              [14,[15,16,17,18,19,20,21]]]
short_beta_list_slice=["short_beta_Slice",short_beta_slice]

In [ ]:
def create_splitflow_COM_mapping(main_path, grouping_list: list[str, list[int]]):
    """center of mass coarse grain mapping: creates mapping matrix for SplitFlow and a CG PDB for visual verification."""

    grouping_name = grouping_list[0]
    grouping = grouping_list[1]
    
    # 1. Load Topology and Coordinates
    aa_path=os.path.join(main_path,"config/data/ala2/ala2.pdb")
    traj = md.load(aa_path)
    top = traj.topology
    n_atoms = top.n_atoms       #get the number of atoms in the topology
    n_beads = len(grouping)     #get the number of beads of the predefined coarse grained grouping
    
    # 2. Initialize the Mapping Matrix (Shape: Atoms x Beads)
    # SplitFlow calls: self.map_matrix.T @ x
    # (Atoms x Beads).T @ (Atoms x 3)=(Beads x Atoms) @ (Atoms x 3) = (Beads x 3)
    mapping_matrix = np.zeros((n_atoms, n_beads), dtype=np.float32)
    
    # 3. Fill matrix with Mass-Weighted values
    for bead_idx, atom_indices in enumerate(grouping):
        masses = np.array([top.atom(i).element.mass for i in atom_indices])
        total_mass = np.sum(masses)
        
        for i, atom_idx in enumerate(atom_indices):
            mapping_matrix[atom_idx, bead_idx] = masses[i] / total_mass     

    # 4. Save the pure matrix for SplitFlow
    matrix_tensor = torch.from_numpy(mapping_matrix)
    #save as a pytorch tensor .pt
    matrix_tensor_path=os.path.join(main_path,f"config/data/mapping_matrices/ala2_{grouping_name}.pt")
    torch.save(matrix_tensor, matrix_tensor_path)
    print(f"Saved mapping matrix")



    # 5. Create a Visualization PDB
    # We calculate the COM positions for the first frame [0]
    fg_coords = traj.xyz[0] # Shape (22 dialanine Atoms, 3 xyz)
    # Manual COM calculation matching the matrix logic:
    cg_coords = mapping_matrix.T @ fg_coords # Shape (Beads, 3)         #this is also done in the SplitFlow
    
    # Create a simple CG topology for the PDB visualization (for simplicity made up of carbon, since masses are not taken into account in backmapping)
    cg_top = md.Topology()      
    chain = cg_top.add_chain()      #create new molecule topology with chain as bead container
    for i in range(n_beads):
        res = cg_top.add_residue(f"BD{i+1}", chain)       #several beads called BD1,BD2 etc. inside the add_chain container
        cg_top.add_atom(f"BEAD", md.element.carbon, res)    #add a representative carbon atom at each bead position
    
    #optional: adding the bonds for a simple cg chain
    for i in range(n_beads - 1):
        atom1 = cg_top.atom(i)
        atom2 = cg_top.atom(i + 1)
        cg_top.add_bond(atom1, atom2)

    # Save the CG PDB
    cg_traj = md.Trajectory(cg_coords[np.newaxis, :], cg_top)       #create the new beads at the positions of the calculated COMs
    #np.newaxis, because previously deleted axis by traj.xyz[0]
    matrix_pdb_path=os.path.join(main_path,f"config/data/ala2/ala2_{grouping_name}.pdb")
    cg_traj.save_pdb(matrix_pdb_path)
    print(f"Saved visualization PDB")



def create_splitflow_sliced_mapping(main_path, grouping_list: list[str, list[int, list]]):          
    """sliced coarse grain mapping such that the atom group bead is mapped onto the position of the central atom. Creates .pdb file"""

    grouping_name = grouping_list[0]
    grouping=grouping_list[1]
    
    #create new coarsened molecule topology
    cg_top = md.Topology()
    chain=cg_top.add_chain()
    for i in range(len(grouping)):
        res = cg_top.add_residue(f"BD{i+1}",chain)
        cg_top.add_atom(f"BEAD",md.element.carbon,res)

    # 1. Load Topology and Coordinates
    aa_path=os.path.join(main_path,"config/data/ala2/ala2.pdb")
    traj=md.load(aa_path)
    aa_coords=traj.xyz[0]
    
    #indices to keep
    idx = []
    for i in grouping:
        idx.append(i[0])
    cg_coords=aa_coords[idx]

    #create a .pdb file
    cg_traj=md.Trajectory(cg_coords[np.newaxis, :], cg_top)
    sliced_pdb_path=os.path.join(main_path,f"config/data/ala2/ala2_{grouping_name}.pdb")
    cg_traj.save_pdb(sliced_pdb_path)
    print("Saved visualization PDB")

    #save the latent_groupings in a json file to be accessed by .yaml
    latent_dir=os.path.join(main_path,"config/data/latent_groupings")
    os.makedirs(latent_dir,exist_ok=True)
    json_path = os.path.join(latent_dir, f"ala2_{grouping_name}.json")

    with open(json_path, 'w') as f:
        # We save only the grouping list (the nested indices), not the name
        json.dump(grouping, f)
    print("saved latent_grouping as .json")

In [7]:
create_splitflow_COM_mapping(
    main_path=main_path,
    grouping_list=short_beta_list_COM
)

Saved mapping matrix
Saved visualization PDB


In [8]:
create_splitflow_sliced_mapping(
    main_path=main_path,
    grouping_list=short_beta_list_slice
)

Saved visualization PDB
saved latent_grouping as .json


Visualizing the coarse grained molecule using VMD and compare it to the all atom configuration:

In [9]:
import subprocess, platform

In [10]:
def open_vmd(main_path,aa_pdb, cg_pdb=None):

    if platform.system == "Windows":
        # Standard path for VMD on Windows; adjust if your installation is elsewhere
        vmd_path = r"C:\Program Files (x86)\University of Illinois\VMD\vmd.exe"
    else: #for linux users
        vmd_path = "vmd"
    
    aa_pdb_path=os.path.join(main_path, aa_pdb)
    if not os.path.exists(aa_pdb_path):
        print(f"Error: {aa_pdb_path} not found.")
        return

    # Command: vmd -m aa_file.pdb cg_file.pdb
    cmd = [vmd_path, aa_pdb_path]
    if cg_pdb:
        cg_pdb_path=os.path.join(main_path, cg_pdb)
        if not os.path.exists(cg_pdb_path):
            print(f"Error: {cg_pdb_path} not found.")
            return
        cmd.append(cg_pdb_path)
    
    subprocess.Popen(cmd)

# Usage
open_vmd(main_path,"config/data/ala2/ala2.pdb", "config/data/ala2/ala2_ramachandran_beta_Slice.pdb") 